# M.E.R.L.I.N. — QLoRA Fine-Tuning v2 (Qwen 2.5-1.5B)

Fine-tune Qwen 2.5-1.5B-Instruct pour generer nativement le format **VERBE — description**.

**Hardware**: Google Colab Free Tier (T4 16GB)
**Method**: QLoRA (4-bit NF4 + LoRA r=16, 7 modules)
**Dataset**: `merlin_verbs_v5_augmented.jsonl` (1734 samples, ChatML `messages` key)
**Output**: GGUF Q4_K_M + F16 pour Ollama

## 1. Installation

In [ ]:
%%capture
!pip install unsloth
# Unsloth installe automatiquement: transformers, peft, bitsandbytes, trl, accelerate
!pip install llama-cpp-python  # Pour conversion GGUF
print("Installation terminee.")

## 2. Upload du dataset

Uploader `merlin_verbs_v5_augmented.jsonl` depuis votre machine locale.

In [ ]:
from google.colab import files
import os

DATASET_PATH = "/content/merlin_verbs_v5_augmented.jsonl"

if not os.path.exists(DATASET_PATH):
    print("Uploadez merlin_verbs_v5_augmented.jsonl...")
    uploaded = files.upload()
    for name in uploaded:
        os.rename(name, DATASET_PATH)
        print(f"Dataset uploade: {name} -> {DATASET_PATH}")
else:
    print(f"Dataset deja present: {DATASET_PATH}")

# Verifier le contenu
import json
samples = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line))
print(f"Samples charges: {len(samples)}")

# v5 uses "messages" key (ChatML standard)
msg_key = "messages" if "messages" in samples[0] else "conversations"
print(f"Format: {msg_key}")
print(f"Exemple (system): {samples[0][msg_key][0]['content'][:100]}...")
print(f"Exemple (user):   {samples[0][msg_key][1]['content'][:100]}...")
print(f"Exemple (asst):   {samples[0][msg_key][2]['content'][:100]}...")

## 3. Charger le modele en 4-bit (QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect (float16 on T4)
    load_in_4bit=True,
)

print(f"Modele charge: {MODEL_NAME}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 4. Configurer LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,  # Unsloth QLoRA: dropout=0 recommande (quantized weights)
    bias="none",
    use_gradient_checkpointing="unsloth",  # Memory-efficient
    random_state=42,
)

# Parametres entrainables
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parametres entrainables: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 5. Preparer le dataset (ChatML)

In [ ]:
from datasets import Dataset
import json

# Charger JSONL
raw_samples = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            raw_samples.append(json.loads(line))

# Detect format key (v5 uses "messages", v1 uses "conversations")
msg_key = "messages" if "messages" in raw_samples[0] else "conversations"

# Convertir en ChatML format texte
def format_chatml(sample):
    convs = sample[msg_key]
    text = ""
    for msg in convs:
        role = msg["role"]
        content = msg["content"]
        text += f"<|im_start|>{role}\n{content}<|im_end|>\n"
    return {"text": text}

formatted = [format_chatml(s) for s in raw_samples]
dataset = Dataset.from_list(formatted)

# Split train/eval (90/10)
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")
print(f"\nExemple formate (premier 500 chars):")
print(train_dataset[0]["text"][:500])

## 6. Entrainement (SFTTrainer)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        # Dataset params (moved from SFTTrainer kwargs in TRL 0.12+)
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_num_proc=2,
        packing=True,
        # Core params
        output_dir="/content/merlin-lora-output",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,
        # Optimizer
        learning_rate=2e-4,
        weight_decay=0.01,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        # Precision
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        # Logging
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        # Misc
        seed=42,
        report_to="none",
    ),
)

print("Demarrage de l'entrainement...")
print(f"  Epochs: 3")
print(f"  Batch size: 4 x 4 gradient accum = 16 effective")
print(f"  Learning rate: 2e-4 (cosine decay)")
print(f"  Warmup: 10%")

In [ ]:
# Lancer l'entrainement
train_result = trainer.train()

# Afficher les metriques
print("\n" + "=" * 50)
print("  RESULTATS D'ENTRAINEMENT")
print("=" * 50)
for key, value in train_result.metrics.items():
    print(f"  {key}: {value}")

## 7. Test rapide (avant export)

In [ ]:
# Passer en mode inference
FastLanguageModel.for_inference(model)

# Test prompts
test_prompts = [
    {
        "system": "Tu es Merlin l'Enchanteur. FORMAT: VERBE — description concrete. Vocabulaire celtique.",
        "user": "Carte 1. Lieu: foret_broceliande. Theme: source sacree. Acte I."
    },
    {
        "system": "Tu es Merlin l'Enchanteur. FORMAT: VERBE — description. Style sensoriel.",
        "user": "Carte 5. Lieu: marais_korrigans. Theme: nuit de Samhain. Corps=bas Ame=equilibre Monde=haut."
    },
    {
        "system": "Tu es Merlin l'Enchanteur. FORMAT: VERBE — description. URGENCE: peril.",
        "user": "Carte 12. Lieu: collines_dolmens. Theme: combat rituel. Acte III. Corps=bas Ame=bas."
    },
]

import re

for i, prompt in enumerate(test_prompts):
    chatml = (
        f"<|im_start|>system\n{prompt['system']}<|im_end|>\n"
        f"<|im_start|>user\n{prompt['user']}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(chatml, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.3,
        do_sample=True,
    )
    result = tokenizer.decode(outputs[0], skip_special_tokens=False)
    # Extract assistant response
    match = result.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0]
    
    print(f"\n{'='*60}")
    print(f"  TEST {i+1}: {prompt['user'][:60]}...")
    print(f"{'='*60}")
    print(match.strip())
    
    # Quick format check
    verb_pattern = r'^[A-D]\)\s+[A-Z].*[—–\-].*'
    lines = match.strip().split('\n')
    verb_lines = [l for l in lines if re.match(verb_pattern, l.strip())]
    print(f"\n  Format check: {len(verb_lines)}/3 lignes VERBE — description")

## 8. Export GGUF (pour Ollama)

In [ ]:
# Sauvegarder le modele merge (LoRA fusionné dans le base model)
MERGED_DIR = "/content/merlin-qwen-1.5b-merged"
GGUF_DIR = "/content/merlin-qwen-1.5b-gguf"

print("Merge LoRA weights into base model...")
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)
print(f"Modele merge sauvegarde: {MERGED_DIR}")

In [ ]:
# Export GGUF Q4_K_M (optimal pour CPU inference via Ollama)
print("Export GGUF Q4_K_M...")
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"GGUF Q4_K_M exporte: {GGUF_DIR}")

# Export F16 GGUF (maximum quality, ~2.9 GB)
GGUF_F16_DIR = "/content/merlin-qwen-1.5b-gguf-f16"
print("Export GGUF F16...")
model.save_pretrained_gguf(
    GGUF_F16_DIR,
    tokenizer,
    quantization_method="f16",
)
print(f"GGUF F16 exporte: {GGUF_F16_DIR}")

# Lister les fichiers generes
import glob
for d in [GGUF_DIR, GGUF_F16_DIR]:
    gguf_files = glob.glob(f"{d}/*.gguf")
    for f in gguf_files:
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  {os.path.basename(f)}: {size_mb:.1f} MB")

## 9. Telecharger le GGUF

In [ ]:
# Telecharger les fichiers GGUF (Q4_K_M + F16)
from google.colab import files
import glob

for d, label in [(GGUF_DIR, "Q4_K_M"), (GGUF_F16_DIR, "F16")]:
    gguf_files = glob.glob(f"{d}/*.gguf")
    if gguf_files:
        gguf_path = gguf_files[0]
        size_mb = os.path.getsize(gguf_path) / (1024 * 1024)
        print(f"Telechargement {label}: {gguf_path} ({size_mb:.0f} MB)")
        files.download(gguf_path)
    else:
        print(f"ERREUR: Aucun fichier GGUF {label} trouve!")

## 10. Instructions post-training

### Deployer dans Ollama

```bash
# 1. Creer un Modelfile
cat > Modelfile <<EOF
FROM ./merlin-qwen-1.5b-Q4_K_M.gguf
PARAMETER temperature 0.75
PARAMETER top_p 0.92
PARAMETER repeat_penalty 1.35
PARAMETER num_ctx 4096
TEMPLATE "<|im_start|>system\n{{ .System }}<|im_end|>\n<|im_start|>user\n{{ .Prompt }}<|im_end|>\n<|im_start|>assistant\n"
SYSTEM "Tu es Merlin l'Enchanteur, druide ancestral de Broceliande."
EOF

# 2. Creer le modele Ollama
ollama create merlin-narrator -f Modelfile

# 3. Tester
ollama run merlin-narrator "Carte 1. Lieu: foret_broceliande. Theme: brume."
```

### Benchmark

```bash
python tools/lora/benchmark_lora.py --results tmp/lora_test_results.json
```

### Metriques cibles

| Metrique | Seuil | Description |
|----------|-------|-------------|
| verb_extraction_rate | > 80% | Lignes A)/B)/C) avec VERBE — desc |
| format_compliance | > 90% | 3 choix correctement formates |
| french_rate | > 85% | Texte en francais |
| verb_diversity (Jaccard) | < 0.5 | Verbes varies (pas toujours les memes) |